# Suite XRX — The Cpp RegExp dialect

`Expressions.Cpp.RegExp` models the regular expressions `std::regex` reads (ISO/IEC 14882, [re.grammar] and
[re.synopt]), one dialect for its six grammars: ECMAScript's (ECMA-262, third edition, as [re.grammar] modifies it),
POSIX's basic and extended regular expressions, and those of awk, grep and egrep. A `regex` names its grammar among its
`syntax_option_type` flags; its kinds are the constructs the grammars share, and which grammar has a construct is
checked by validation. No position of C++ holds one, a pattern being a string there: its holder does. Its kinds are
validated, bind nothing, cover C++'s three editions, print in each grammar, are refused by evaluation by name, and are
checked complete against its grammars kept as data.

In [ ]:
import { readFileSync as readGrammar } from "node:fs";
import { Framework } from "@mbse/expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as CFamily from "@mbse/expressions/C/Expressions";
import * as CPreprocessor from "@mbse/expressions/C/Preprocessor";
import * as CTypes from "@mbse/expressions/C/Types";
import * as CppFamily from "@mbse/expressions/Cpp/Expressions";
import * as CppTypes from "@mbse/expressions/Cpp/Types";
import * as CppRegExp from "../src/Cpp/RegExp/index.js";
import { Evaluators as FE, Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Validators } from "@mbse/schemas/Framework";
import { assert, equal, raises, same } from "./support.js";

const { NotImplementedError, ValueError } = Errors;
const [PP, CPP, XT, XR, XRText] = [CPreprocessor.Expressions, CppFamily.Expressions, CppTypes.Expressions, CppRegExp.Expressions, CppRegExp.Text];

const store: any = new Builders.Store(); // the C family's dialects, Cpp RegExp's install first: the installs run in any order
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, CppRegExp.install, CFamily.install, CppFamily.install,
  CTypes.install, CppTypes.install, CPreprocessor.install]) {
  store.transaction(install);
}
store.transaction(CppRegExp.define); // its editions

/** The term value `spec` writes, the root of `dialect`'s holder, written in a transaction. */
function term(spec: unknown, dialect: F.Dialect = XR.DIALECT): any {
  return F.term(store.transaction(() => dialect.path.split(".").reduce((at: any, part) => at[part], store.Expressions)
    .Expression((e: any) => e.root(dialect.spec(spec)))));
}

const validate = (spec: unknown): string[] => XR.DIALECT.validate(term(spec));
const text = (spec: unknown, grammar = "ECMAScript"): string => XRText.text(term(spec), grammar);
const union = (name: string): string[] => store.registered(name).branches.map((b: any) => b.name); // a union's branches
/** A character, of its code point, and its spelling. */
const char = (c: string, spelling?: string) => (u: any) => u.character(
  (r: any) => spelling === undefined ? r.value(BigInt(c.codePointAt(0) as number)) : r.raw(spelling).value(BigInt(c.codePointAt(0) as number)));
const alt = (...elements: unknown[]) => (u: any) => u.alternative((r: any) => r.elements(elements));
const pattern = (...alternatives: unknown[]) => (u: any) => u.pattern((r: any) => r.alternatives(alternatives));
const capture = (...alternatives: unknown[]) => (u: any) => u.capturinggroup((r: any) => r.alternatives(alternatives));
const group = (...alternatives: unknown[]) => (u: any) => u.group((r: any) => r.alternatives(alternatives));

/** A quantifier: `element{least,most}`, lazy where not `greedy`, as `spelling` writes it where given. */
function repeat(element: unknown, least: bigint, most?: bigint, greedy?: boolean, spelling?: string): unknown {
  return (u: any) => u.quantifier((r: any) => {
    r = r.min(least).element(element);
    r = most === undefined ? r : r.max(most);
    r = greedy === undefined ? r : r.greedy(greedy);
    return spelling === undefined ? r : r.raw(spelling);
  });
}

const cls = (elements: unknown[], negate = false) => (u: any) => u.characterclass((r: any) => r.negate(negate).elements(elements));
const span = (low: unknown, high: unknown) => (u: any) => u.characterclassrange((r: any) => r.min(low).max(high));
const chars = (kind: string, negate = false) => (u: any) => u.characterset((r: any) => r.setKind(kind).negate(negate));
const anchor = (kind: string, negate = false) => (u: any) => u.assertion((r: any) => r.assertionKind(kind).negate(negate));
/** `(?=...)`, `(?!...)`. */
const ahead = (alternatives: unknown[], negate = false) => (u: any) => u.assertion(
  (r: any) => r.assertionKind("lookahead").negate(negate).alternatives(alternatives));
const backref = (n: bigint) => (u: any) => u.backreference((r: any) => r.ref(n));
const named = (kind: string, name: string) => (u: any) => u[kind]((r: any) => r.name(name)); // `[:alpha:]`, `[.a.]`, `[=a=]`
const regex = (flags: string[], spec: unknown) => (u: any) => u.regex((r: any) => r.flags(flags).pattern(spec));
const patterns = (...specs: unknown[]) => (u: any) => u.patternlist((r: any) => r.patterns(specs));
const error = (source: string) => (u: any) => u.error((r: any) => r.text(source));
const missing = (kind: string) => (u: any) => u.missing((r: any) => r.kind(kind));
const abc = pattern(alt(char("a"), char("b")), alt(char("c"))); // `ab|c`

<a id="XRX-01"></a>
## XRX-01 · Its union and holder: its kinds and invalid syntax, a branch of the framework's terms, reading its family's dialects; a holder refining the framework's, round-tripped through JSON

In [ ]:
{
  assert(same(XR.TAGS, ["regex", "patternlist", "pattern", "alternative", "group", "capturinggroup", "quantifier", "characterclass",
    "characterclassrange", "characterset", "character", "assertion", "backreference", "exclass", "collatingelement",
    "equivalence"]) && XR.HELPERS.length === 0 && XR.SINCE.size === 0);
  assert(same([...XR.DIALECT.kinds().keys()], union(XR.DIALECT.union)) && same(union(XR.DIALECT.union), [...XR.TAGS, "error", "missing"]));
  assert(XR.DIALECT.name() === "Cpp RegExp" && XR.DIALECT.tag === "cpp_regexp" && XR.DIALECT.union === "Expressions.Cpp.RegExp.Term");
  assert(union("Expressions.Framework.Term").includes("cpp_regexp") && union("Expressions.Framework.Dialect").includes("cpp_regexp"));
  assert(same(XR.DIALECT.reads(), [CPP.DIALECT, XT.DIALECT, PP.DIALECT])); // its family's dialects
  assert(!XR.DIALECT.reads().some((dialect) => XR.TAGS.some((tag) => union(dialect.union).includes(tag)))); // no position holds one
  const Holder: any = store.registered("Expressions.Cpp.RegExp.Expression");
  assert(Holder.ref && Holder.refines === store.registered("Expressions.Framework.Expression") && F.isExpression(Holder));
  assert(Holder.properties.get("root").schema === store.registered(XR.DIALECT.union) && same([...Holder.adjacencies.keys()], ["abi"]));
  const holder = store.transaction(() => store.Expressions.Cpp.RegExp.Expression((e: any) => e.root(XR.DIALECT.spec(regex(["extended", "icase"], abc)))));
  assert(Validators.Validate(store)(Holder, holder).length === 0 && XR.DIALECT.validate(holder).length === 0);
  assert(F.dialectOf(F.term(holder)) === XR.DIALECT.union && F.kindOf(F.term(holder))[0] === "regex");
  assert(F.same((JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, holder)) as any).root, holder.root));
  raises(Error, () => store.transaction(() => // its root holds its own terms
    store.Expressions.Cpp.RegExp.Expression((e: any) => e.root(XR.DIALECT.spec(CPP.binary("+", CPP.identifier("x"), 1n))))),
  "'binaryexpression' is not a member of a union value");
  assert(XR.DIALECT.validate(term(CPP.identifier("x"), CPP.DIALECT), ["x"]).length === 0); // a C++ term, as C++ reads it
}

<a id="XRX-02"></a>
## XRX-02 · Its kinds: their properties in order, each term property its Term; forms and make round-trip

In [ ]:
{
  const props = (name: string): string[] => [...store.registered(`Expressions.Cpp.RegExp.${name}`).properties.keys()];
  assert(same(props("Regex"), ["flags", "pattern"]) && same(props("PatternList"), ["patterns"]));
  assert(same(props("Pattern"), ["alternatives"]) && same(props("Group"), ["alternatives"]) && same(props("CapturingGroup"), ["alternatives"])
    && same(props("Alternative"), ["elements"]));
  assert(same(props("Quantifier"), ["raw", "min", "max", "greedy", "element"]) && same(props("CharacterClass"), ["negate", "elements"]));
  assert(same(props("CharacterClassRange"), ["min", "max"]) && same(props("CharacterSet"), ["setKind", "negate"]));
  assert(same(props("Character"), ["raw", "value"]) && same(props("Assertion"), ["assertionKind", "negate", "alternatives"]));
  assert(same(props("Backreference"), ["ref"]) && same(props("ExClass"), ["name"]) && same(props("CollatingElement"), ["name"])
    && same(props("Equivalence"), ["name"]));
  const prop = (name: string, member: string): any => store.registered(`Expressions.Cpp.RegExp.${name}`).properties.get(member).schema;
  const Term = store.registered(XR.DIALECT.union);
  assert(prop("Regex", "pattern") === Term && prop("Quantifier", "element") === Term && prop("CharacterClassRange", "max") === Term);
  assert(prop("Pattern", "alternatives").element === Term && prop("Alternative", "elements").element === Term
    && prop("Assertion", "alternatives").element === Term);
  const made = F.form(term(regex(["basic"], pattern(alt(repeat(char("a"), 2n, 3n), backref(1n))))));
  assert(made.kind === "regex" && same((made.attributes.get("flags") as any).values, ["basic"])
    && same(made.arguments.map((a) => F.kindOf(a)[0]), ["pattern"]));
  const quantified = F.form(term(repeat(cls([span(char("a"), char("z"))]), 1n, undefined, false)));
  assert(equal(quantified.attributes, new Map<string, unknown>([["min", 1n], ["greedy", false]]))
    && same(quantified.arguments.map((a) => F.kindOf(a)[0]), ["characterclass"]));
  for (const spec of [regex(["grep"], patterns(abc, pattern(alt(chars("any"))))), ahead([alt(anchor("word", true))], true),
    cls([named("exclass", "alpha"), named("collatingelement", "ch"), named("equivalence", "e"), chars("digit")], true)]) {
    assert(F.same(term(XR.DIALECT.make(F.form(term(spec)))), term(spec)));
  }
}

<a id="XRX-03"></a>
## XRX-03 · Its checks: what each kind's terms may be, and its attributes' values

In [ ]:
{
  const cases: [unknown, string[]][] = [
    [regex([], alt(char("a"))), ["a regex's pattern cannot hold an alternative"]],
    [patterns(alt(char("a"))), ["a patternlist's patterns cannot hold an alternative"]],
    [pattern(char("a")), ["a pattern's alternatives cannot hold a character"]],
    [alt(pattern(alt())), ["an alternative's elements cannot hold a pattern"]],
    [group(alt(named("exclass", "alpha"))), ["argument 0: an alternative's elements cannot hold an exclass"]],
    [capture(char("a")), ["a capturinggroup's alternatives cannot hold a character"]],
    [repeat(alt(), 0n), ["a quantifier's element cannot hold an alternative"]],
    [cls([alt()]), ["a characterclass's elements cannot hold an alternative"]],
    [span(chars("digit"), named("collatingelement", "z")), ["a characterclassrange's min cannot hold a characterset"]],
    [ahead([char("a")]), ["an assertion's alternatives cannot hold a character"]],
    [repeat(char("a"), -1n), ["a quantifier's min cannot be negative, got -1"]],
    [repeat(char("a"), 3n, 2n), ["a quantifier's max cannot be less than its min, got 2 and 3"]],
    [repeat(anchor("word"), 0n), ["a quantifier cannot repeat a word assertion"]],
    [(u: any) => u.quantifier((r: any) => r.element(char("a"))), ["a quantifier needs a min"]],
    [(u: any) => u.quantifier((r: any) => r.min(1n)), ["a quantifier needs an element"]],
    [chars("alpha"), ["CharacterSet.setKind cannot be 'alpha'"]],
    [chars("any", true), ["an any characterset cannot be negated"]],
    [(u: any) => u.characterset((r: any) => r), ["a characterset needs a setKind"]],
    [(u: any) => u.character((r: any) => r.value(0x110000n)), ["a character's value must be a code point, got 1114112"]],
    [(u: any) => u.character((r: any) => r.raw("a")), ["a character needs a value"]],
    [anchor("both"), ["Assertion.assertionKind cannot be 'both'"]],
    [anchor("start", true), ["a start assertion cannot be negated"]],
    [(u: any) => u.assertion((r: any) => r.assertionKind("end").alternatives([alt()])), ["an end assertion holds no alternatives"]],
    [(u: any) => u.assertion((r: any) => r.negate(true)), ["an assertion needs an assertionKind"]],
    [backref(0n), ["a backreference's ref must be positive, got 0"]],
    [named("exclass", "a:b"), ["an exclass's name cannot hold ':'"]],
    [named("collatingelement", "a.="), ["a collatingelement's name cannot hold '.'"]],
    [named("equivalence", ""), ["an equivalence needs a name"]],
  ];
  for (const [spec, problems] of cases) assert(same(validate(spec), problems), `${validate(spec)} != ${problems}`);
  for (const spec of [abc, alt(), cls([span(char("a"), named("collatingelement", "z"))]), repeat(ahead([alt(char("a"))]), 0n, 1n),
    cls([chars("word", true), char("-"), named("equivalence", "e")])]) {
    assert(validate(spec).length === 0);
  }
}

<a id="XRX-04"></a>
## XRX-04 · Its grammars: a regex's flags, its one grammar element, ECMAScript by default; what each grammar has, checked

In [ ]:
{
  assert(same(XR.GRAMMARS, ["ECMAScript", "basic", "extended", "awk", "grep", "egrep"]) && XR.ECMASCRIPT === "ECMAScript");
  assert(same(XR.SYNTAX_OPTIONS, ["icase", "nosubs", "optimize", "collate", ...XR.GRAMMARS, "multiline"]));
  assert(XR.grammar(["icase", "awk"]) === "awk" && XR.grammar(["icase"]) === "ECMAScript" && XR.grammar([]) === "ECMAScript"); // the default
  const unflagged = (u: any) => u.regex((r: any) => r.pattern(abc)); // a regex with no flags, ECMAScript's
  assert(validate(unflagged).length === 0 && XRText.ToText(term(unflagged)) === "ab|c");
  assert(same(validate(regex(["icase", "verbose", "basic"], abc)), ["'verbose' is not a syntax_option_type element",
    "alternation is not in the basic grammar"]));
  assert(same(validate(regex(["basic", "extended"], abc)), ["a regex has at most one grammar element, got basic, extended"]));
  assert(same(validate(regex(["basic"], missing("pattern"))), ["pattern: missing pattern"])
    && same(validate((u: any) => u.regex((r: any) => r.flags(["grep"]))), ["a regex needs a pattern"]));
  assert(same(XR.FEATURES.get("alternation")!, ["ECMAScript", "extended", "awk", "egrep"]) && same(XR.FEATURES.get("a group")!, ["ECMAScript"]));
  const ecma = pattern(alt(group(alt(char("a"))), ahead([alt(char("b"))]), anchor("word"), chars("digit"), chars("space"),
    chars("word", true), repeat(char("c"), 0n, undefined, false), backref(12n)));
  assert(validate(regex([], ecma)).length === 0 && validate(regex(["ECMAScript", "multiline"], ecma)).length === 0);
  assert(same(validate(regex(["extended"], ecma)), [
    "a group is not in the extended grammar", "a lookahead assertion is not in the extended grammar",
    "a word assertion is not in the extended grammar", "a digit characterset is not in the extended grammar",
    "a space characterset is not in the extended grammar", "a word characterset is not in the extended grammar",
    "a lazy quantifier is not in the extended grammar", "a backreference is not in the extended grammar",
    "a backreference past 9 is not in the extended grammar"]));
  assert(same(validate(regex(["grep"], pattern(alt(cls([chars("digit")]), backref(9n))))), ["a digit characterset is not in the grep grammar"]));
  const anchored = pattern(alt(anchor("start"), capture(alt(anchor("start"), char("a"))), anchor("end")), alt(anchor("end"), char("b")));
  assert(validate(regex(["egrep"], anchored)).length === 0 && validate(regex(["awk"], anchored)).length === 0);
  assert(same(validate(regex(["basic"], anchored)), ["alternation is not in the basic grammar",
    "an anchor within the expression is not in the basic grammar"]));
  assert(validate(regex(["basic"], pattern(alt(anchor("start"), char("a"), anchor("end"))))).length === 0); // where a BRE begins and ends
  const repeated = pattern(alt(repeat(repeat(char("a"), 1n), 0n, 1n), repeat(anchor("start"), 0n)));
  assert(validate(regex(["extended"], repeated)).length === 0);
  assert(same(validate(regex(["ECMAScript"], repeated)), ["a repeated quantifier is not in the ECMAScript grammar",
    "a repeated anchor is not in the ECMAScript grammar"]));
  assert(same(validate(regex(["extended"], pattern(alt((u: any) => u.quantifier((r: any) => r.min(1n)))))), [
    "pattern: argument 0: argument 0: a quantifier needs an element"]));
  assert(validate(regex(["grep"], patterns(pattern(alt(char("a"))), pattern(alt(anchor("start")))))).length === 0); // each line's own
  assert(validate(regex(["egrep"], patterns(abc))).length === 0 && same(validate(regex(["awk"], patterns(abc))), [
    "a patternlist is not in the awk grammar"]));
  assert(same(validate(regex(["basic"], pattern(alt(repeat(missing("atom"), 1n), error("\\{"))))), [ // invalid syntax: its own
    "pattern: argument 0: argument 0: element: missing atom", "pattern: argument 0: argument 1: invalid syntax '\\\\{'"]));
}

<a id="XRX-05"></a>
## XRX-05 · What it binds: nothing; a capturing group numbered, a backreference resolving it by its number

In [ ]:
{
  assert(!store.registered(XR.DIALECT.union).branches.some((b: any) => F.isBinder(b.schema)));
  const grouped = term(regex([], pattern(alt(capture(alt(char("a"))), backref(1n)))));
  assert(SY.free(grouped, XR.DIALECT).size === 0 && XR.DIALECT.validate(grouped).length === 0);
  assert(same([...F.walk(grouped)].map((t) => F.kindOf(t)[0]), ["regex", "pattern", "alternative", "capturinggroup", "alternative",
    "character", "backreference"]));
}

<a id="XRX-06"></a>
## XRX-06 · Invalid syntax: error and missing in its Term, validated, printed as written, refused by name

In [ ]:
{
  assert(same(validate(error("(?<")), ["invalid syntax '(?<'"]) && text(error("(?<")) === "(?<");
  assert(same(validate(missing("atom")), ["missing atom"]) && text(alt(char("a"), missing("atom"))) === "a");
  assert(same(validate(pattern(alt(char("a"), error("[")))), ["argument 0: argument 1: invalid syntax '['"]));
  assert(text(pattern(alt(char("a"), error("[z")))) === "a[z");
  raises(ValueError, () => text((u: any) => u.error((r: any) => r)), "Cpp RegExp error has no text to print");
  raises(ValueError, () => XRText.ToText(term(CPP.identifier("x"), CPP.DIALECT)), "Cpp idexpression is not a regular expression to print");
}

<a id="XRX-07"></a>
## XRX-07 · Its editions: C++'s three, each by its designation and common name, each with every kind

In [ ]:
{
  const regexp = store.Instances.Expressions.Cpp.RegExp;
  assert(equal([...FS.standards(regexp).values()].map((s) => FS.tags(s)), [
    ["ISO/IEC 14882:2017", "C++17"], ["ISO/IEC 14882:2020", "C++20"], ["ISO/IEC 14882:2024", "C++23"]]));
  assert(equal(FS.properties(FS.find(regexp, "C++17")!).get("revision"), ["year", 2017n]));
  const names = ["Regex", "PatternList", "Pattern", "Alternative", "Group", "CapturingGroup", "Quantifier", "CharacterClass",
    "CharacterClassRange", "CharacterSet", "Character", "Assertion", "Backreference", "ExClass", "CollatingElement", "Equivalence"];
  assert([...FS.standards(regexp).values()].every((e) => same(FS.kinds(e).map((s: any) => s.name),
    names.map((name) => `Expressions.Cpp.RegExp.${name}`))));
  assert(store.transaction(CppRegExp.define)[0].get("Cpp20") === FS.find(regexp, "C++20")); // run again, it writes nothing
}

<a id="XRX-08"></a>
## XRX-08 · Printing: every kind in each grammar, its escapes, a text as written

In [ ]:
{
  const every = pattern(alt(anchor("start"), capture(alt(char("a")), alt(char("."))), repeat(char("b"), 0n), repeat(char("c"), 1n),
    repeat(char("d"), 0n, 1n), repeat(char("e"), 2n, 2n), repeat(char("f"), 2n), repeat(char("g"), 2n, 5n),
    cls([span(char("a"), char("z")), named("exclass", "digit"), named("collatingelement", "ch"), named("equivalence", "e"), char("]")], true),
    chars("any"), backref(1n), anchor("end")));
  for (const [grammar, source] of [
    ["ECMAScript", "^(a|\\.)b*c+d?e{2}f{2,}g{2,5}[^a-z[:digit:][.ch.][=e=]\\]].\\1$"],
    ["basic", "^\\(a|\\.\\)b*c\\{1,\\}d\\{0,1\\}e\\{2\\}f\\{2,\\}g\\{2,5\\}[^a-z[:digit:][.ch.][=e=]]].\\1$"],
    ["grep", "^\\(a|\\.\\)b*c\\{1,\\}d\\{0,1\\}e\\{2\\}f\\{2,\\}g\\{2,5\\}[^a-z[:digit:][.ch.][=e=]]].\\1$"],
    ["extended", "^(a|\\.)b*c+d?e{2}f{2,}g{2,5}[^a-z[:digit:][.ch.][=e=]]].\\1$"],
    ["awk", "^(a|\\.)b*c+d?e{2}f{2,}g{2,5}[^a-z[:digit:][.ch.][=e=]]].\\1$"],
  ] as [string, string][]) {
    assert(text(regex([grammar], every)) === source && text(every, grammar) === source, `${grammar}: ${text(regex([grammar], every))}`);
  }
  const ecma = pattern(alt(group(alt(char("a"))), ahead([alt(char("b"))]), ahead([alt(char("c"))], true), anchor("word"),
    anchor("word", true), chars("digit"), chars("space", true), chars("word"), repeat(char("x"), 0n, undefined, false),
    repeat(char("y"), 1n, 3n, true)),
  alt(char("\n"), char("\x01"), char("\x7f"), char("é"), cls([char("^"), char("-"), char("\\"), char("\t"), char("a")])));
  assert(XRText.ToText(term(ecma)) === "(?:a)(?=b)(?!c)\\b\\B\\d\\S\\wx*?y{1,3}|\\n\\x01\\x7fé[\\^\\-\\\\\\ta]");
  assert(text(regex(["egrep"], patterns(abc, pattern(alt(char("(")))))) === "ab|c\n\\(");
  assert(text(regex(["awk"], pattern(alt(char("\n"), char("\x07"), char("\x08"), char("\x01"), char("\x7f"), char("/"), char("+"),
    cls([char("\t"), char("+")]))))) === "\\n\\a\\b\\001\\177/\\+[\\t+]");
  assert(text(regex(["basic"], pattern(alt(char("+"), char("*"), char("\n"), cls([char("\\")]))))) === "+\\*\n[\\]");
  assert(text(regex(["basic"], pattern(alt(repeat(char("a"), 1n, undefined, undefined, "\\{1,\\}"), char("x", "\\x"))))) === "a\\{1,\\}\\x"); // as written
  assert(text(repeat(char("a"), 0n, undefined, undefined, "{0,}")) === "a{0,}" && text(char("A", "\\x41")) === "\\x41");
  assert(XRText.ToText(term(regex(["icase"], abc))) === "ab|c"); // its flags are basic_regex's constructor's other argument
}

<a id="XRX-09"></a>
## XRX-09 · Evaluation: each kind refused by its dialect's name

In [ ]:
{
  const interpreter = new FE.Interpreter(XR.DIALECT, new Map(), { evaluated: new Set() }); // no kind evaluated, each refused by its dialect's name
  for (const spec of [regex([], abc), abc, char("a"), cls([]), backref(1n)]) {
    raises(NotImplementedError, () => interpreter.run(term(spec)), `Cpp RegExp ${F.kindOf(term(spec))[0]} is not evaluated`);
  }
  raises(ValueError, () => interpreter.run(term(error("("))), "Cpp RegExp error is not evaluated");
}

<a id="XRX-10"></a>
## XRX-10 · Completeness: grammars/cpp-regex.json maps every production of its grammars, and every kind is mapped

In [ ]:
{
  const grammar = globalThis.JSON.parse(readGrammar("../../grammars/cpp-regex.json", "utf8"));
  const nodes: any[] = grammar.nodes;
  assert(nodes.length === 78 && new Set(nodes.map((n) => `${n.source}|${n.node}`)).size === 78 && grammar.checked.includes("78 productions and tokens"));
  assert(nodes.every((n) => {
    const keys = Object.keys(n).filter((k) => k !== "note").sort().join(",");
    return keys === "kinds,node,source" || keys === "node,not,source";
  }));
  const mapped = new Set(nodes.flatMap((n) => n.kinds ?? []));
  assert(equal([...mapped].sort(), XR.TAGS.map((tag) => `RegExp.${tag}`).sort())); // every kind, exactly: invalid syntax the framework's
  const sources: string[] = nodes.map((n) => n.source);
  assert(same([...new Set(sources)], [
    "ISO/IEC 14882", "ECMA-262 3rd edition, 15.10.1, as [re.grammar] modifies it", "ISO/IEC/IEEE 9945, XBD 9.5",
    "ISO/IEC/IEEE 9945, XCU awk, Regular Expressions", "ISO/IEC/IEEE 9945, XCU grep"]));
  const added = ["ClassAtomExClass", "ClassAtomCollatingElement", "ClassAtomEquivalence", "ClassName", "ClassNameCharacter"];
  assert(added.every((name) => nodes.some((n) => n.node === name))); // [re.grammar]'s own productions
}